NMPC utilizando RNN com SciPy

In [33]:
from tensorflow.keras.models import load_model

model = load_model("model_JacketedTank.h5", compile=False)

In [34]:
import numpy as np

# Função Dinâmica via RNN

WINDOW = 6
N_FEATURES = 4  # [u1, u2, y1, y2]

def rnn_dynamics(history, u):
    """
    history: (6, 4)
    u: (2,)
    """
    print("history shape:", history.shape)
    # última saída conhecida
    y_last = history[-1, 2:]

    # novo vetor de entrada
    new_row = np.concatenate([u, y_last])

    # atualiza janela
    new_history = np.vstack([history[1:], new_row])

    # reshape pro modelo
    inp = new_history.reshape(1, 6, 4) #(1, WINDOW, N_FEATURES)
    print("Shape entrada:", inp.shape)
    y_next = model.predict(inp, verbose=0)

    return y_next.flatten(), new_history

In [35]:
# Simulação no Horizonte

def simulate_trajectory(history, U, N):
    Y = []
    h = history.copy()

    for k in range(N):
        y, h = rnn_dynamics(h, U[k])
        Y.append(y)

    return np.array(Y)

In [36]:
# Função Custo

def cost_function(U, history, ref, N):

    if np.isnan(U).any() or np.isinf(U).any():
        return 1e10  # penalização forte

    U = U.reshape(N, 2)  # 2 entradas

    history_local = history.copy()
    Y = simulate_trajectory(history_local, U, N)

    J = 0
    for k in range(N):
        # erro nas 2 saídas
        J += np.sum((Y[k] - ref[k])**2)

        # penalização controle
        J += 0.01*np.sum(U[k]**2)

    return J    

In [37]:
from scipy.optimize import minimize

# Otimização NMPC

def solve_nmpc(history, ref, N, u_init):
    bounds = [(-1, 1)] * (N * 2)
    res = minimize(
        cost_function,
        u_init.flatten(),
        args=(history, ref, N),
        method='SLSQP',
        bounds=bounds
    )

    return res.x.reshape(N, 2)

In [40]:
# Loop NMPC

'''
Fazer:
u_norm = (u - mean_u) / std_u
y_norm = (y - mean_y) / std_y
'''

N = 4
window_size = 6

'''
# histórico inicial (IMPORTANTE!)
history = np.zeros((window_size, 4))  
# ajuste nº de features conforme seu modelo

u_init = np.zeros((N, 2))
'''
history = np.zeros((6,4), dtype=np.float32)
u = np.array([0.1, 0.2], dtype=np.float32)
traj = []

for t in range(50):

    ref = np.zeros((N, 2))  # referência 2 saídas

    U_opt = solve_nmpc(history, ref, N, u_init)

    u = U_opt[0]

    y, history = rnn_dynamics(history, u)

    traj.append(y)

    u_init = U_opt  # warm start

history shape: (6, 4)
Shape entrada: (1, 6, 4)


ValueError: Exception encountered when calling SimpleRNNCell.call().

[1mDimensions must be equal, but are 4 and 2 for '{{node sequential_1/simple_rnn_1/simple_rnn_cell_1/MatMul}} = MatMul[T=DT_FLOAT, grad_a=false, grad_b=false, transpose_a=false, transpose_b=false](sequential_1/simple_rnn_1/strided_slice_1, sequential_1/simple_rnn_1/simple_rnn_cell_1/Cast/ReadVariableOp)' with input shapes: [1,4], [2,64].[0m

Arguments received by SimpleRNNCell.call():
  • sequence=tf.Tensor(shape=(1, 4), dtype=float32)
  • states=('tf.Tensor(shape=(1, 64), dtype=float32)',)
  • training=False

# Híbrido com CasADi

- Lento e Preciso

In [ ]:
import casadi as ca

N = 10

# variável de controle
U = ca.MX.sym('U', N, 2)

# parâmetro: histórico inicial
H0 = ca.MX.sym('H0', WINDOW, N_FEATURES)

# referência
REF = ca.MX.sym('REF', N, 2)

J = 0

h = H0

for k in range(N):

    u_k = U[k,:]

    # ⚠️ chamada externa (black-box)
    y_k, h = rnn_dynamics(h, u_k)

    # custo
    J += ca.sumsqr(y_k - REF[k,:])
    J += 0.01*ca.sumsqr(u_k)

# NLP
nlp = {
    'x': ca.reshape(U, -1, 1),
    'p': ca.vertcat(ca.reshape(H0, -1, 1),
                    ca.reshape(REF, -1, 1)),
    'f': J
}

solver = ca.nlpsol('solver', 'ipopt', nlp)

Exception: Implicit conversion of symbolic CasADi type to numeric matrix not supported.
This may occur when you pass a CasADi object to a numpy function.
Use an equivalent CasADi function instead of that numpy function.

In [ ]:
def solve_casadi(history, ref, u_init):

    p = np.concatenate([
        history.flatten(),
        ref.flatten()
    ])

    sol = solver(
        x0=u_init.flatten(),
        p=p
    )

    U_opt = sol['x'].full().reshape(N, 2)
    return U_opt

# Puro CasADi

- Rápido e Preciso

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("model_JacketedTank.h5", compile=False)
weights = model.get_weights()

In [ ]:
# camada 1
Wx1, Wh1, b1 = weights[0], weights[1], weights[2]
# camada 2
Wx2, Wh2, b2 = weights[3], weights[4], weights[5]
# dense
W_dense, b_dense = weights[6], weights[7]

In [ ]:
import casadi as ca
import numpy as np

def rnn_casadi():

    # entrada: sequência (6 passos, 4 features)
    X = ca.MX.sym('X', 6, 4)

    # estados ocultos
    h1 = ca.MX.zeros(64)
    h2 = ca.MX.zeros(32)

    for t in range(6):

        x_t = X[t, :]

        # camada 1
        h1 = ca.tanh(
            ca.mtimes(x_t, Wx1) +
            ca.mtimes(h1, Wh1) +
            b1
        )

        # camada 2
        h2 = ca.tanh(
            ca.mtimes(h1, Wx2) +
            ca.mtimes(h2, Wh2) +
            b2
        )

    # saída
    y = ca.mtimes(h2, W_dense) + b_dense

    return ca.Function('rnn', [X], [y])

In [ ]:
rnn_fun = rnn_casadi()

# entrada aleatória
x_test = np.random.randn(1,6,4)

y_keras = model.predict(x_test)
y_casadi = rnn_fun(x_test[0]).full()

print(y_keras, y_casadi)

RuntimeError: .../casadi/core/sparsity.cpp:432: Assertion "x.size2()==y.size1()" failed:
Matrix product with incompatible dimensions. Lhs is 1x4 and rhs is 2x64.

In [ ]:
N = 10

U = ca.MX.sym('U', N, 2)
X_hist = ca.MX.sym('X_hist', 6, 4)
REF = ca.MX.sym('REF', N, 2)

J = 0
h = X_hist

for k in range(N):

    u_k = U[k,:]

    # última saída
    y_last = h[-1, 2:]

    new_row = ca.hcat([u_k, y_last])

    h = ca.vertcat(h[1:,:], new_row)

    y = rnn_fun(h)

    J += ca.sumsqr(y - REF[k,:])
    J += 0.01*ca.sumsqr(u_k)

In [ ]:
nlp = {
    'x': ca.reshape(U, -1, 1),
    'p': ca.vertcat(
        ca.reshape(X_hist, -1, 1),
        ca.reshape(REF, -1, 1)
    ),
    'f': J
}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Adicionar Métricas de Desempenho 

- MSE
- Tempo de computação
- Esforço de controle $\sum u^2$
- Estabilidade